## GPT END-TO-END

Next Word Prediction

GPT uses:

* Decoder Only
* Causal Attention
* Predict Next Token

In [25]:
data = [
    "start i am a student end",
    "start how are you end",
    "start i love machine learning end",
    "start good morning end",
    "start thank you end",
    "start see you later end",
    "start what is your name end",
    "start where are you going end",
    "start i like coffee end",
    "start welcome end"
]

In [26]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention
)

from tensorflow.keras import Model

In [27]:
vocab_size = 1000
sequence_length = 20

vectorizer = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

vectorizer.adapt(data)

tokens = vectorizer(data)

inputs = tokens[:, :-1]
targets = tokens[:, 1:]

In [28]:
class PositionalEmbedding(tf.keras.layers.Layer):

    def __init__(
        self,
        sequence_length,
        vocab_size,
        embed_dim
    ):
        super().__init__()

        self.token_embedding = Embedding(
            vocab_size,
            embed_dim
        )

        self.position_embedding = Embedding(
            sequence_length,
            embed_dim
        )

    def call(self, inputs):

        length = tf.shape(inputs)[-1]

        positions = tf.range(
            start=0,
            limit=length,
            delta=1
        )

        embedded_tokens = self.token_embedding(
            inputs
        )

        embedded_positions = self.position_embedding(
            positions
        )

        return embedded_tokens + embedded_positions

In [29]:
class GPTDecoder(tf.keras.layers.Layer):

    def __init__(
        self,
        embed_dim,
        dense_dim,
        num_heads
    ):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            Dense(
                dense_dim,
                activation="relu"
            ),
            Dense(embed_dim)
        ])

        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()

    def call(self, inputs):

        attention_output = self.attention(
            query=inputs,
            value=inputs,
            key=inputs,
            use_causal_mask=True
        )

        out1 = self.layernorm1(
            inputs + attention_output
        )

        ffn_output = self.ffn(out1)

        return self.layernorm2(
            out1 + ffn_output
        )

In [30]:
embed_dim = 128
dense_dim = 256
num_heads = 4

gpt_input = tf.keras.Input(
    shape=(None,),
    dtype="int64"
)

x = PositionalEmbedding(
    sequence_length,
    vocab_size,
    embed_dim
)(gpt_input)

x = GPTDecoder(
    embed_dim,
    dense_dim,
    num_heads
)(x)

output = Dense(
    vocab_size,
    activation="softmax"
)(x)

gpt = Model(
    gpt_input,
    output
)

In [33]:
gpt.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

gpt.fit(
    inputs,
    targets,
    batch_size=2,
    epochs=30
)

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9316 - loss: 0.1815
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9526 - loss: 0.1535 
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9526 - loss: 0.1429 
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9526 - loss: 0.1365 
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9474 - loss: 0.1347 
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9474 - loss: 0.1326 
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9526 - loss: 0.1296 
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9526 - loss: 0.1301 
Epoch 9/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9526 - loss: 0.1284
Epoch 10/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9526 - loss: 0.1284 
Epoch 11/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9526 - loss: 0.1283 
Epoch 12/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9474 - loss: 0.1274 
Ep

In [35]:
prompt = "i like"

generated = prompt

for i in range(5):

    tokenized = vectorizer(
        [generated]
    )

    predictions = gpt.predict(
        tokenized,
        verbose=0
    )

    length = np.count_nonzero(
    tokenized.numpy()[0]
    )

    next_token_id = np.argmax(
    predictions[0, length-1, :]
    )

    vocab = vectorizer.get_vocabulary()

    next_word = vocab[next_token_id]

    generated += " " + next_word

print(generated)

i like coffee end   
